## Gradient Descent (경사하강법)

4-1에서 $w$에 따른 비용 곡선을 그려 보았습니다. 곡선의 **가장 낮은 지점**을 찾으면 최적의 $w$를 얻을 수 있습니다.

* 정규 방정식: 공식으로 한 번에 계산 → 모델이 복잡해지면 공식을 구할 수 없음
* **경사하강법**: 현재 위치에서 기울기(gradient)를 보고, 비용이 줄어드는 방향으로 조금씩 이동
    - 뒤에서 배울 DNN, CNN, RNN 모두 경사하강법으로 학습함

In [ ]:
from IPython.display import Image
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/10-1.GD.png', width=500)

#### 1. 아이디어

$$
gradient = \frac{\partial L(w)}{\partial w}
$$

* 기울기가 양수일 때 (Positive gradient): $w$를 **감소**시켜야 비용이 줄어듦
* 기울기가 음수일 때 (Negative gradient): $w$를 **증가**시켜야 비용이 줄어듦
* 두 경우 모두 **기울기의 반대 방향**으로 이동 → 하나의 식으로 표현 가능

<center>

$w := w - \alpha \times \frac{\partial L}{\partial w}$

</center>

* $\alpha$: 학습률(learning rate), 한 번에 얼마나 이동할지 결정하는 값

#### 2. 데이터 준비 (4-1과 동일)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
hours = np.round(rng.uniform(1, 10, 20), 1)
score = np.round(5 * hours + 40 + rng.normal(0, 5, 20), 1)

x = hours
y = score

def predict(x, w, b):
    return w * x + b

def mse(y, y_pred):
    return np.mean((y - y_pred) ** 2)

#### 3. 기울기(gradient) 구하기

비용 함수

<center>

$L(w,b)= \frac{1}{n}\sum_{i=1}^{n} (\hat{y}^{(i)}-y^{(i)})^2 = \frac{1}{n}\sum_{i=1}^{n} (wx^{(i)} + b-y^{(i)})^2$

</center>

를 $w$와 $b$로 각각 편미분 (합성함수 미분 이용)

<center>

$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}^{(i)}-y^{(i)})\, x^{(i)}$

$\frac{\partial L}{\partial b} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}^{(i)}-y^{(i)})$

</center>

In [ ]:
def gradient(x, y, w, b):
    y_pred = predict(x, w, b)
    error = y_pred - y
    dw = 2 * np.mean(error * x)
    db = 2 * np.mean(error)
    return dw, db

In [ ]:
# w=0, b=0 에서의 기울기
gradient(x, y, 0, 0)

* 두 기울기 모두 음수 → $w$와 $b$를 **증가**시켜야 비용이 줄어든다는 의미

#### 4. 경사하강법 학습 루프

1. 예측값 $\hat{y}$ 계산
2. 비용(loss) 계산
3. 기울기 계산
4. $w$, $b$ 업데이트

위 과정을 여러 번 반복. 전체 데이터를 한 번 사용하는 주기를 **에포크(epoch)** 라고 함

In [ ]:
w, b = 0.0, 0.0        # 초기값
lr = 0.01              # 학습률
n_epochs = 3000

loss_history = []
for epoch in range(n_epochs + 1):
    # 1) 예측
    y_pred = predict(x, w, b)
    # 2) loss 계산
    loss = mse(y, y_pred)
    loss_history.append(loss)
    # 3) 기울기 계산
    dw, db = gradient(x, y, w, b)
    # 4) w, b 업데이트
    w = w - lr * dw
    b = b - lr * db

    if epoch % 300 == 0:
        print(f'Epoch {epoch:4d}/{n_epochs}  w: {w:.3f}  b: {b:.3f}  Loss: {loss:.4f}')

In [ ]:
# 4-1의 정규 방정식 결과와 비교
w_best = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
b_best = y.mean() - w_best * x.mean()
print(f'경사하강법  : w = {w:.3f}, b = {b:.3f}')
print(f'정규 방정식 : w = {w_best:.3f}, b = {b_best:.3f}')

In [ ]:
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.yscale('log')   # loss가 급격히 줄어들기 때문에 로그 스케일로 표시
plt.show()

In [ ]:
x_line = np.linspace(0, 11, 100)
plt.scatter(x, y, c='steelblue', edgecolor='white', s=70)
plt.plot(x_line, predict(x_line, w, b), color='black', lw=2)
plt.xlabel('Study hours')
plt.ylabel('Score')
plt.show()

#### 5. 학습 과정 시각화

$b$를 고정하고 $w$만 학습시키면서, 4-1의 비용 곡선 위에서 $w$가 어떻게 이동하는지 확인

In [ ]:
b_fixed = 40
w_list = np.linspace(-1, 11, 200)
cost_list = [mse(y, predict(x, w_, b_fixed)) for w_ in w_list]

w = 0.0
lr = 0.005
w_path = [w]
for epoch in range(10):
    dw, _ = gradient(x, y, w, b_fixed)
    w = w - lr * dw
    w_path.append(w)

w_path = np.array(w_path)
plt.plot(w_list, cost_list, label='cost')
plt.plot(w_path, [mse(y, predict(x, w_, b_fixed)) for w_ in w_path], 'ro-', label='GD steps')
plt.xlabel('w')
plt.ylabel('MSE')
plt.legend()
plt.show()

* 기울기가 가파른 곳에서는 크게, 최저점 근처에서는 작게 이동 ($\alpha \times gradient$ 이므로)

#### 6. 학습률 (Learning rate)

* 학습률이 **너무 작으면**: 학습이 매우 느림
* 학습률이 **너무 크면**: 최저점을 지나쳐 **발산**(diverge)할 수 있음

In [ ]:
def train(x, y, lr, n_epochs):
    w, b = 0.0, 0.0
    loss_history = []
    for epoch in range(n_epochs):
        loss_history.append(mse(y, predict(x, w, b)))
        dw, db = gradient(x, y, w, b)
        w = w - lr * dw
        b = b - lr * db
    return w, b, loss_history

In [ ]:
plt.figure(figsize=(12, 4))
for i, lr in enumerate([0.0005, 0.01, 0.03]):
    w, b, history = train(x, y, lr, n_epochs=50)
    plt.subplot(1, 3, i + 1)
    plt.plot(history)
    plt.title(f'lr = {lr}')
    plt.xlabel('Epoch')
    plt.ylabel('MSE')
plt.tight_layout()
plt.show()

* `lr = 0.0005`: loss가 천천히 감소
* `lr = 0.01`: 빠르게 수렴
* `lr = 0.03`: loss가 오히려 폭발적으로 증가 (발산)

In [ ]:
# lr = 0.03 일 때 w가 최저점을 넘나들며 점점 멀어지는 모습
b_fixed = 40
w = 0.0
lr = 0.03
w_path = [w]
for epoch in range(5):
    dw, _ = gradient(x, y, w, b_fixed)
    w = w - lr * dw
    w_path.append(w)
print(np.round(w_path, 2))

#### 7. 파이토치와의 연결 (6장 미리보기)

위에서 직접 작성한 학습 루프는 6장에서 파이토치 코드로 다음과 같이 바뀜

| 단계 | NumPy (직접 구현) | PyTorch |
|---|---|---|
| 예측 | `y_pred = w * x + b` | `y_pred = model(x)` |
| loss 계산 | `loss = np.mean((y_pred - y) ** 2)` | `loss = F.mse_loss(y_pred, y)` |
| 기울기 계산 | `dw, db = gradient(x, y, w, b)` (직접 미분) | `optimizer.zero_grad()`<br>`loss.backward()` (자동 미분) |
| 업데이트 | `w = w - lr * dw`<br>`b = b - lr * db` | `optimizer.step()` |

```python
# 6-2에서 배울 파이토치 코드
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(n_epochs):
    y_pred = model(x_train)               # 1) 예측
    loss = F.mse_loss(y_pred, y_train)    # 2) loss 계산
    optimizer.zero_grad()
    loss.backward()                       # 3) 기울기 계산 (자동 미분)
    optimizer.step()                      # 4) w, b 업데이트
```

* 모델이 복잡해지면 미분 공식을 손으로 구하기 어려움 → 파이토치가 **자동으로 미분**해 줌

In [ ]:
## 실습
# 1) 학습률을 0.001, 0.005, 0.02로 바꾸어 3000 에포크 학습한 뒤, 최종 w, b와 loss를 비교하세요.
# 2) 초기값을 w=10, b=0으로 바꾸어도 같은 결과로 수렴하는지 확인하세요.